# BeautyCore optimized scheduled demo

Separate release notebook. Keep the original starter as fallback. Use the same pinned Kaggle image, T4, Internet, both existing private datasets and the same three enabled Secrets. Run the three code cells once in order. New source is installed only after original bundle/environment/model checks. Nails-only public-prompt reuse; Hair/Makeup remain on baseline. No model quality settings are reduced. First live browser/GPU rehearsal remains required.


In [ ]:
"""Verified private bundles and early host preflight for the client demo notebook."""
from pathlib import Path
from hashlib import sha256
from zipfile import ZipFile
import json, platform, socket, sys, uuid

BUNDLES = (
    ("capstone_train002_20260928_v1.bin", "f0d1fed52e00b6c5a1da2cade3731237675cdebbfc0b2cde3b2dbda99a4019ac", "gpu-src", "gate3_bundle.json"),
    ("render_kaggle_cpu_20261009.bin", "f391050851519e9cc1080830b615a6cc457b6dd1e21fde156522bb959fad9ede", "cpu-src", "demo_bundle.json"),
)

def extract(source, expected, target, manifest_name):
    if sha256(source.read_bytes()).hexdigest() != expected:
        raise RuntimeError("Bundle hash differs: " + source.name)
    if target.exists():
        raise RuntimeError("Extraction requires a new directory; existing source is never overwritten")
    with ZipFile(source) as archive:
        manifest = json.loads(archive.read(manifest_name))
        names = archive.namelist()
        if len(names) != len(set(names)) or set(names) != set(manifest["files"]) | {manifest_name}:
            raise RuntimeError("Bundle member list differs")
        if archive.testzip() is not None:
            raise RuntimeError("Bundle archive integrity failed")
        # Validate every member before writing any of the archive contents.
        members = []
        for name in names:
            destination = (target / name).resolve()
            if ":" in name or chr(92) in name or not destination.is_relative_to(target.resolve()):
                raise RuntimeError("Unsafe bundle member path")
            raw = archive.read(name)
            if name != manifest_name and sha256(raw).hexdigest() != manifest["files"][name]:
                raise RuntimeError("Bundle member hash differs: " + name)
            members.append((destination, raw))
        for destination, raw in members:
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(raw)

def check_host(observed, expected):
    if any(observed.get(key) != expected.get(key) for key in ("python", "torch", "cuda", "gpu")):
        raise RuntimeError("Kaggle image differs from the reviewed GPU image. Expected "
                           + json.dumps(expected) + "; observed " + json.dumps(observed)
                           + ". Use the tested pinned notebook; do not change the model guards.")

def prepare(inputs=Path("/kaggle/input"), working=Path("/kaggle/working")):
    if sys.platform != "linux":
        raise RuntimeError("Run this starter inside Kaggle, not on the client device")
    for port in (8765, 8800):
        with socket.socket() as connection:
            connection.settimeout(1)
            if connection.connect_ex(("127.0.0.1", port)) == 0:
                raise RuntimeError("An AI service is already running. Do not run setup again.")
    work = working / ("render-kaggle-client-" + uuid.uuid4().hex)
    for name, digest, directory, manifest in BUNDLES:
        matches = list(inputs.rglob(name))
        if len(matches) != 1:
            raise RuntimeError("Attach exactly one private input containing " + name)
        extract(matches[0], digest, work / directory, manifest)
    import torch
    observed = {"python": platform.python_version(), "torch": torch.__version__,
                "cuda": torch.version.cuda,
                "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
    review = json.loads((work / "gpu-src/docs/experiments/unified-kaggle-gate2-review.json").read_text())
    expected = {key: review["environment"][key] for key in observed}
    check_host(observed, expected)
    print("Private bundles and reviewed GPU host verified.", flush=True)
    print("Working directory:", work, flush=True)
    print("Run Cell 2 once. Keep this Kaggle session running during the demo.", flush=True)
    return work

CLIENT_WORK = prepare()
"""Stage the reviewed source overlay in a fresh extracted demo workspace only."""
import base64
from hashlib import sha256
import json
from pathlib import Path
import socket

FILES = frozenset({'scripts/inference_performance.py', 'scripts/unified_gate3_server.py'})

def validated(payload):
    if not isinstance(payload, dict) or set(payload) != FILES:
        raise ValueError('Unexpected optimization source inventory')
    result = {}
    for name, item in payload.items():
        if not isinstance(item, dict) or set(item) != {'data', 'sha256'}:
            raise ValueError('Invalid optimization source record')
        raw = base64.b64decode(item['data'], validate=True)
        if sha256(raw).hexdigest() != item['sha256']:
            raise ValueError('Optimization source hash differs: ' + name)
        compile(raw, name, 'exec')
        result[name] = raw
    return result

def stage_overlay(work, payload):
    work = Path(work)
    validated(payload)
    target = work / 'optimized-worker.json'
    with target.open('x', encoding='utf-8') as file:
        json.dump(payload, file)
    return target

def install_overlay(root, payload_file, output):
    root, output = Path(root), Path(output)
    for port in (8765, 8800):
        with socket.socket() as connection:
            connection.settimeout(1)
            if connection.connect_ex(('127.0.0.1', port)) == 0:
                raise RuntimeError('Do not replace source while an AI service is running')
    record = output / 'optimized-worker-release.json'
    if record.exists():
        raise RuntimeError('Optimization already installed; use a fresh session')
    payload = json.loads(Path(payload_file).read_text(encoding='utf-8'))
    sources = validated(payload)
    manifest_file = root / 'gate3_bundle.json'
    manifest = json.loads(manifest_file.read_text(encoding='utf-8'))
    # The original bootstrap verifies the entire bundle first. Check every replaced
    # file again here; never rewrite that pinned manifest or private archive.
    for name in FILES:
        target = root / name
        if name in manifest['files']:
            if sha256(target.read_bytes()).hexdigest() != manifest['files'][name]:
                raise ValueError('Original worker source differs before overlay')
        elif target.exists():
            raise ValueError('Unexpected preexisting optimization source')
    for name, raw in sources.items():
        target = root / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(raw)
    evidence = {'policy': 'nails-only', 'original_manifest_sha256': sha256(manifest_file.read_bytes()).hexdigest(),
                'source_sha256': {name: sha256(raw).hexdigest() for name, raw in sources.items()},
                'model_parameters_changed': False}
    record.write_text(json.dumps(evidence, indent=2), encoding='utf-8')
    print('OPTIMIZATION: Nails-only public-prompt reuse; Hair/Makeup use baseline inference.', flush=True)
    return evidence

OPTIMIZED_SOURCES = {'scripts/inference_performance.py': {'data': 'IiIiT3B0LWluIHBoYXNlIHRpbWluZyBhbmQgYm91bmRlZCBmaXhlZC1wcm9tcHQgcmV1c2U7IG5vIGltYWdlL3Jlc3VsdCBjYWNoZS4NCg0KRGVmYXVsdCBzZXJ2aW5nIGRvZXMgbm90IGluc3RhbGwgd3JhcHBlcnMuIEEgY2FuZGlkYXRlIG11c3QgcGFzcyBtYXRjaGVkIEdQVQ0KcXVhbGl0eS90aW1pbmcgcmV2aWV3IGJlZm9yZSBlbmFibGluZyBBSV9QUk9NUFRfQ0FDSEUgaW4gYSBkZXBsb3ltZW50Lg0KIiIiDQpmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBPcmRlcmVkRGljdA0KZnJvbSBjb250ZXh0bGliIGltcG9ydCBjb250ZXh0bWFuYWdlcg0KZnJvbSBoYXNobGliIGltcG9ydCBzaGEyNTYNCmltcG9ydCBpbnNwZWN0DQppbXBvcnQganNvbg0KaW1wb3J0IHRpbWUNCg0KDQpjbGFzcyBJbmZlcmVuY2VQZXJmb3JtYW5jZToNCiAgICBkZWYgX19pbml0X18oc2VsZiwgc3luY2hyb25pemUsICosIGNhY2hlX2VuYWJsZWQ9RmFsc2UsIG1heF9lbnRyaWVzPTQsIG1heF9ieXRlcz02NCoxMDI0KjEwMjQsIHBoYXNlX3RpbWluZz1UcnVlLCBjbG9jaz10aW1lLm1vbm90b25pYywgY2FjaGVfZmVhdHVyZXM9Tm9uZSk6DQogICAgICAgIGlmIG5vdCAxIDw9IG1heF9lbnRyaWVzIDw9IDE2IG9yIG5vdCAxIDw9IG1heF9ieXRlcyA8PSAxMjgqMTAyNCoxMDI0Og0KICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignSW52YWxpZCBwcm9tcHQgY2FjaGUgYm91bmRzJykNCiAgICAgICAgc2VsZi5zeW5jaHJvbml6ZSwgc2VsZi5jbG9jayA9IHN5bmNocm9uaXplLCBjbG9jaw0KICAgICAgICBzZWxmLmNhY2hlX2VuYWJsZWQsIHNlbGYubWF4X2VudHJpZXMsIHNlbGYubWF4X2J5dGVzID0gY2FjaGVfZW5hYmxlZCwgbWF4X2VudHJpZXMsIG1heF9ieXRlcw0KICAgICAgICBzZWxmLmNhY2hlX2ZlYXR1cmVzID0gTm9uZSBpZiBjYWNoZV9mZWF0dXJlcyBpcyBOb25lIGVsc2UgZnJvemVuc2V0KGNhY2hlX2ZlYXR1cmVzKQ0KICAgICAgICBzZWxmLmNhY2hlID0gT3JkZXJlZERpY3QoKQ0KICAgICAgICBzZWxmLmNhY2hlX2J5dGVzID0gMA0KICAgICAgICBzZWxmLnBoYXNlX3RpbWluZyA9IHBoYXNlX3RpbWluZw0KICAgICAgICBzZWxmLmxhc3QgPSBOb25lDQogICAgICAgIHNlbGYuX21lYXN1cmluZyA9IEZhbHNlDQoNCiAgICBkZWYgY2FjaGVfZm9yKHNlbGYsIGZlYXR1cmUpOg0KICAgICAgICByZXR1cm4gc2VsZi5jYWNoZV9lbmFibGVkIGFuZCAoc2VsZi5jYWNoZV9mZWF0dXJlcyBpcyBOb25lIG9yIGZlYXR1cmUgaW4gc2VsZi5jYWNoZV9mZWF0dXJlcykNCg0KICAgIGRlZiBhcHBsaWVzX3RvKHNlbGYsIGZlYXR1cmUpOg0KICAgICAgICAjIEhhaXIvTWFrZXVwIGJ5cGFzcyB0aGUgd3JhcHBlciBhbmQgaXRzIHN5bmNocm9uaXphdGlvbnMgZW50aXJlbHkgaW4gTmFpbHMtb25seSBzZXJ2aW5nLg0KICAgICAgICByZXR1cm4gc2VsZi5waGFzZV90aW1pbmcgb3Igc2VsZi5jYWNoZV9mb3IoZmVhdHVyZSkNCg0KICAgIGRlZiBjbGVhcihzZWxmKToNCiAgICAgICAgc2VsZi5jYWNoZS5jbGVhcigpDQogICAgICAgIHNlbGYuY2FjaGVfYnl0ZXMgPSAwDQoNCiAgICBAc3RhdGljbWV0aG9kDQogICAgZGVmIF9ieXRlcyh0ZW5zb3JzKToNCiAgICAgICAgcmV0dXJuIHN1bSh2YWx1ZS5udW1lbCgpKnZhbHVlLmVsZW1lbnRfc2l6ZSgpIGZvciB2YWx1ZSBpbiB0ZW5zb3JzKQ0KDQogICAgZGVmIF9rZXkoc2VsZiwgc2lnbmF0dXJlLCBhcmdzLCBrd2FyZ3MsIGlkZW50aXR5LCBhbGxvd2VkKToNCiAgICAgICAgYm91bmQgPSBzaWduYXR1cmUuYmluZCgqYXJncywgKiprd2FyZ3MpDQogICAgICAgIGJvdW5kLmFwcGx5X2RlZmF1bHRzKCkNCiAgICAgICAgdmFsdWVzID0gYm91bmQuYXJndW1lbnRzDQogICAgICAgIHByb21wdCA9IHZhbHVlcy5nZXQoJ3Byb21wdCcpDQogICAgICAgIGlmIHZhbHVlcy5nZXQoJ3Byb21wdF9lbWJlZHMnKSBpcyBub3QgTm9uZSBvciBub3QgaXNpbnN0YW5jZShwcm9tcHQsIHN0cikgb3IgcHJvbXB0IG5vdCBpbiBhbGxvd2VkOg0KICAgICAgICAgICAgcmV0dXJuIE5vbmUNCiAgICAgICAgIyBPbmx5IGRlY2xhcmVkIHB1YmxpYyBwcm9tcHRzIGNhbiBiZSBjYWNoZWQ7IGltYWdlL3VzZXIgYXJndW1lbnRzIG5ldmVyIGVudGVyIGEga2V5Lg0KICAgICAgICBpZiBzZXQodmFsdWVzKSAtIHsncHJvbXB0JywnZGV2aWNlJywnbnVtX2ltYWdlc19wZXJfcHJvbXB0JywncHJvbXB0X2VtYmVkcycsJ21heF9zZXF1ZW5jZV9sZW5ndGgnLCd0ZXh0X2VuY29kZXJfb3V0X2xheWVycyd9Og0KICAgICAgICAgICAgcmV0dXJuIE5vbmUNCiAgICAgICAgb3B0aW9ucyA9IHtrZXk6c3RyKHZhbHVlKSBmb3Iga2V5LHZhbHVlIGluIHZhbHVlcy5pdGVtcygpIGlmIGtleSBub3QgaW4gKCdwcm9tcHQnLCdwcm9tcHRfZW1iZWRzJyl9DQogICAgICAgIHJldHVybiAoaWRlbnRpdHksc2hhMjU2KHByb21wdC5lbmNvZGUoKSkuaGV4ZGlnZXN0KCksanNvbi5kdW1wcyhvcHRpb25zLHNvcnRfa2V5cz1UcnVlKSkNCg0KICAgIEBjb250ZXh0bWFuYWdlcg0KICAgIGRlZiBtZWFzdXJlKHNlbGYsIHBpcGUsICosIGlkZW50aXR5LCBhbGxvd2VkX3Byb21wdHMsIGZlYXR1cmU9Tm9uZSk6DQogICAgICAgIGlmIHNlbGYuX21lYXN1cmluZzoNCiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignQ29uY3VycmVudCBwZXJmb3JtYW5jZSBzY29wZSByZWZ1c2VkJykNCiAgICAgICAgc2VsZi5fbWVhc3VyaW5nID0gVHJ1ZQ0KICAgICAgICBwaGFzZV9zZWNvbmRzLCBjb3VudHMgPSB7fSwge30NCiAgICAgICAgc3RhdHMgPSB7J2hpdHMnOjAsJ21pc3Nlcyc6MCwnYnlwYXNzZXMnOjB9DQogICAgICAgIHJlc3RvcmUgPSBbXQ0KICAgICAgICBtYXJrZXIgPSBvYmplY3QoKQ0KICAgICAgICBzZWxmLmxhc3QgPSBOb25lDQoNCiAgICAgICAgZGVmIHdyYXBwZWQodGFyZ2V0LCBuYW1lLCBwaGFzZSwgcHJvbXB0PUZhbHNlKToNCiAgICAgICAgICAgIGlmIChub3Qgc2VsZi5waGFzZV90aW1pbmcgYW5kIG5vdCBwcm9tcHQpIG9yIG5vdCBoYXNhdHRyKHRhcmdldCxuYW1lKToNCiAgICAgICAgICAgICAgICByZXR1cm4NCiAgICAgICAgICAgIG9yaWdpbmFsID0gZ2V0YXR0cih0YXJnZXQsbmFtZSkNCiAgICAgICAgICAgIHByZXZpb3VzID0gdmFycyh0YXJnZXQpLmdldChuYW1lLG1hcmtlcikNCiAgICAgICAgICAgIHNpZ25hdHVyZSA9IGluc3BlY3Quc2lnbmF0dXJlKG9yaWdpbmFsKSBpZiBwcm9tcHQgZWxzZSBOb25lDQogICAgICAgICAgICBkZWYgaW52b2tlKCphcmdzLCoqa3dhcmdzKToNCiAgICAgICAgICAgICAgICBpZiBzZWxmLnBoYXNlX3RpbWluZzoNCiAgICAgICAgICAgICAgICAgICAgc2VsZi5zeW5jaHJvbml6ZSgpDQogICAgICAgICAgICAgICAgc3RhcnRlZCA9IHNlbGYuY2xvY2soKQ0KICAgICAgICAgICAgICAgIHRyeToNCiAgICAgICAgICAgICAgICAgICAgaWYgbm90IHByb21wdCBvciBub3Qgc2VsZi5jYWNoZV9mb3IoZmVhdHVyZSk6DQogICAgICAgICAgICAgICAgICAgICAgICByZXR1cm4gb3JpZ2luYWwoKmFyZ3MsKiprd2FyZ3MpDQogICAgICAgICAgICAgICAgICAgIGtleSA9IHNlbGYuX2tleShzaWduYXR1cmUsYXJncyxrd2FyZ3MsKGlkZW50aXR5LHN0cihnZXRhdHRyKHBpcGUsJ2R0eXBlJyxOb25lKSksc3RyKGdldGF0dHIoZ2V0YXR0cihwaXBlLCd0ZXh0X2VuY29kZXInLE5vbmUpLCdkdHlwZScsTm9uZSkpLGlkKGdldGF0dHIocGlwZSwndGV4dF9lbmNvZGVyJyxwaXBlKSksc3RyKGdldGF0dHIocGlwZSwnX2V4ZWN1dGlvbl9kZXZpY2UnLE5vbmUpKSksYWxsb3dlZF9wcm9tcHRzKQ0KICAgICAgICAgICAgICAgICAgICBpZiBrZXkgaXMgTm9uZToNCiAgICAgICAgICAgICAgICAgICAgICAgIHN0YXRzWydieXBhc3NlcyddICs9IDENCiAgICAgICAgICAgICAgICAgICAgICAgIHJldHVybiBvcmlnaW5hbCgqYXJncywqKmt3YXJncykNCiAgICAgICAgICAgICAgICAgICAgaWYga2V5IGluIHNlbGYuY2FjaGU6DQogICAgICAgICAgICAgICAgICAgICAgICBjcHUsZGV2aWNlcyxzaXplID0gc2VsZi5jYWNoZS5wb3Aoa2V5KQ0KICAgICAgICAgICAgICAgICAgICAgICAgc2VsZi5jYWNoZVtrZXldID0gKGNwdSxkZXZpY2VzLHNpemUpDQogICAgICAgICAgICAgICAgICAgICAgICBzdGF0c1snaGl0cyddICs9IDENCiAgICAgICAgICAgICAgICAgICAgICAgIHJldHVybiB0dXBsZSh2YWx1ZS50byhkZXZpY2U9ZGV2aWNlKS5jbG9uZSgpIGZvciB2YWx1ZSxkZXZpY2UgaW4gemlwKGNwdSxkZXZpY2VzKSkNCiAgICAgICAgICAgICAgICAgICAgc3RhdHNbJ21pc3NlcyddICs9IDENCiAgICAgICAgICAgICAgICAgICAgdmFsdWUgPSBvcmlnaW5hbCgqYXJncywqKmt3YXJncykNCiAgICAgICAgICAgICAgICAgICAgIyBlbmNvZGVfcHJvbXB0IHJldHVybnMgdGhlIHR3byB0ZW5zb3JzIChlbWJlZGRpbmdzLCB0ZXh0IElEcykuDQogICAgICAgICAgICAgICAgICAgIGlmIG5vdCBpc2luc3RhbmNlKHZhbHVlLHR1cGxlKSBvciBsZW4odmFsdWUpIT0yIG9yIG5vdCBhbGwoaGFzYXR0cih2LCdkZXRhY2gnKSBmb3IgdiBpbiB2YWx1ZSk6DQogICAgICAgICAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ1Vuc3VwcG9ydGVkIHByb21wdCBlbmNvZGVyIHJldHVybiBjb250cmFjdCcpDQogICAgICAgICAgICAgICAgICAgIHNpemUgPSBzZWxmLl9ieXRlcyh2YWx1ZSkNCiAgICAgICAgICAgICAgICAgICAgaWYgc2l6ZSA8PSBzZWxmLm1heF9ieXRlczoNCiAgICAgICAgICAgICAgICAgICAgICAgIGNwdSA9IHR1cGxlKHYuZGV0YWNoKCkudG8oZGV2aWNlPSdjcHUnKS5jbG9uZSgpIGZvciB2IGluIHZhbHVlKQ0KICAgICAgICAgICAgICAgICAgICAgICAgd2hpbGUgc2VsZi5jYWNoZSBhbmQgKGxlbihzZWxmLmNhY2hlKT49c2VsZi5tYXhfZW50cmllcyBvciBzZWxmLmNhY2hlX2J5dGVzK3NpemU+c2VsZi5tYXhfYnl0ZXMpOg0KICAgICAgICAgICAgICAgICAgICAgICAgICAgIF8sKF8sXyxyZW1vdmVkKT1zZWxmLmNhY2hlLnBvcGl0ZW0obGFzdD1GYWxzZSkNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBzZWxmLmNhY2hlX2J5dGVzIC09IHJlbW92ZWQNCiAgICAgICAgICAgICAgICAgICAgICAgIHNlbGYuY2FjaGVba2V5XT0oY3B1LHR1cGxlKHYuZGV2aWNlIGZvciB2IGluIHZhbHVlKSxzaXplKQ0KICAgICAgICAgICAgICAgICAgICAgICAgc2VsZi5jYWNoZV9ieXRlcyArPSBzaXplDQogICAgICAgICAgICAgICAgICAgIHJldHVybiB2YWx1ZQ0KICAgICAgICAgICAgICAgIGZpbmFsbHk6DQogICAgICAgICAgICAgICAgICAgIGlmIHNlbGYucGhhc2VfdGltaW5nOg0KICAgICAgICAgICAgICAgICAgICAgICAgc2VsZi5zeW5jaHJvbml6ZSgpDQogICAgICAgICAgICAgICAgICAgICAgICBwaGFzZV9zZWNvbmRzW3BoYXNlXSA9IHBoYXNlX3NlY29uZHMuZ2V0KHBoYXNlLDApKyhzZWxmLmNsb2NrKCktc3RhcnRlZCkNCiAgICAgICAgICAgICAgICAgICAgICAgIGNvdW50c1twaGFzZV0gPSBjb3VudHMuZ2V0KHBoYXNlLDApKzENCiAgICAgICAgICAgIHNldGF0dHIodGFyZ2V0LG5hbWUsaW52b2tlKQ0KICAgICAgICAgICAgcmVzdG9yZS5hcHBlbmQoKHRhcmdldCxuYW1lLHByZXZpb3VzKSkNCg0KICAgICAgICB0cnk6DQogICAgICAgICAgICBzZWxmLnN5bmNocm9uaXplKCkNCiAgICAgICAgICAgIHN0YXJ0ZWQgPSBzZWxmLmNsb2NrKCkNCiAgICAgICAgICAgIHdyYXBwZWQocGlwZSwnZW5jb2RlX3Byb21wdCcsJ3Byb21wdF9lbmNvZGluZycsVHJ1ZSkNCiAgICAgICAgICAgIHdyYXBwZWQocGlwZSwncHJlcGFyZV9pbWFnZV9sYXRlbnRzJywnaW1hZ2VfZW5jb2RpbmcnKQ0KICAgICAgICAgICAgd3JhcHBlZChwaXBlLnRyYW5zZm9ybWVyLCdmb3J3YXJkJywnZGVub2lzaW5nJykNCiAgICAgICAgICAgIHdyYXBwZWQocGlwZS52YWUsJ2RlY29kZScsJ2ltYWdlX2RlY29kaW5nJykNCiAgICAgICAgICAgIHlpZWxkDQogICAgICAgICAgICBzZWxmLnN5bmNocm9uaXplKCkNCiAgICAgICAgICAgIHNlbGYubGFzdCA9IHsncGhhc2Vfc2Vjb25kcyc6e2tleTpyb3VuZCh2YWx1ZSw2KSBmb3Iga2V5LHZhbHVlIGluIHBoYXNlX3NlY29uZHMuaXRlbXMoKX0sDQogICAgICAgICAgICAgICAgICAgICAgICAgJ3BoYXNlX2NhbGxzJzpjb3VudHMsJ3J1bnRpbWVfc2Vjb25kcyc6cm91bmQoc2VsZi5jbG9jaygpLXN0YXJ0ZWQsNiksDQogICAgICAgICAgICAgICAgICAgICAgICAgJ3Byb21wdF9jYWNoZSc6eyoqc3RhdHMsJ2VuYWJsZWQnOnNlbGYuY2FjaGVfZm9yKGZlYXR1cmUpLCdlbnRyaWVzJzpsZW4oc2VsZi5jYWNoZSksJ2NwdV9ieXRlcyc6c2VsZi5jYWNoZV9ieXRlc30sDQogICAgICAgICAgICAgICAgICAgICAgICAgJ3RpbWluZ19raW5kJzonc3luY2hyb25pemVkX3BoYXNlX2RpYWdub3N0aWMnIGlmIHNlbGYucGhhc2VfdGltaW5nIGVsc2UgJ3dob2xlX3J1bnRpbWVfc3luY2hyb25pemVkX25vX3BoYXNlX2JhcnJpZXJzJ30NCiAgICAgICAgZXhjZXB0IEJhc2VFeGNlcHRpb246DQogICAgICAgICAgICBzZWxmLmNsZWFyKCkNCiAgICAgICAgICAgIHJhaXNlDQogICAgICAgIGZpbmFsbHk6DQogICAgICAgICAgICBmb3IgdGFyZ2V0LG5hbWUscHJldmlvdXMgaW4gcmV2ZXJzZWQocmVzdG9yZSk6DQogICAgICAgICAgICAgICAgaWYgcHJldmlvdXMgaXMgbWFya2VyOg0KICAgICAgICAgICAgICAgICAgICBkZWxhdHRyKHRhcmdldCxuYW1lKQ0KICAgICAgICAgICAgICAgIGVsc2U6DQogICAgICAgICAgICAgICAgICAgIHNldGF0dHIodGFyZ2V0LG5hbWUscHJldmlvdXMpDQogICAgICAgICAgICBzZWxmLl9tZWFzdXJpbmc9RmFsc2UNCg0KDQpkZWYgY29uZmlndXJlZF9wZXJmb3JtYW5jZShzeW5jaHJvbml6ZSwgZW52KToNCiAgICBpZiBlbnYuZ2V0KCdBSV9JTkZFUkVOQ0VfUFJPRklMRScpIT0nMScgYW5kIGVudi5nZXQoJ0FJX1BST01QVF9DQUNIRScpIT0nMSc6DQogICAgICAgIHJldHVybiBOb25lDQogICAgc2NvcGUgPSBlbnYuZ2V0KCdBSV9QUk9NUFRfQ0FDSEVfU0NPUEUnLCAnbmFpbHMnKQ0KICAgIGlmIHNjb3BlIG5vdCBpbiAoJ25haWxzJywgJ2FsbCcpOg0KICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdVbnN1cHBvcnRlZCBwcm9tcHQgY2FjaGUgc2NvcGUnKQ0KICAgIHJldHVybiBJbmZlcmVuY2VQZXJmb3JtYW5jZShzeW5jaHJvbml6ZSxjYWNoZV9lbmFibGVkPWVudi5nZXQoJ0FJX1BST01QVF9DQUNIRScpPT0nMScscGhhc2VfdGltaW5nPWVudi5nZXQoJ0FJX0lORkVSRU5DRV9QUk9GSUxFJyk9PScxJyxjYWNoZV9mZWF0dXJlcz17J25haWxzJ30gaWYgc2NvcGU9PSduYWlscycgZWxzZSBOb25lKQ0K', 'sha256': 'b69dd604baa6a09ba1a1d7598e979c66a81d26f70a8e9ae3db3bb790766030af'}, 'scripts/unified_gate3_server.py': {'data': '...', 'sha256': 'bb1d24159075f0592f491532080195f204f73c708e2a8e2fd095daac481723a2'}}
stage_overlay(CLIENT_WORK, OPTIMIZED_SOURCES)
(CLIENT_WORK/'worker_overlay.py').write_text('"""Stage the reviewed source overlay in a fresh extracted demo workspace only."""\nimport base64\nfrom hashlib import sha256\nimport json\nfrom pathlib import Path\nimport socket\n\nFILES = frozenset({\'scripts/inference_performance.py\', \'scripts/unified_gate3_server.py\'})\n\ndef validated(payload):\n    if not isinstance(payload, dict) or set(payload) != FILES:\n        raise ValueError(\'Unexpected optimization source inventory\')\n    result = {}\n    for name, item in payload.items():\n        if not isinstance(item, dict) or set(item) != {\'data\', \'sha256\'}:\n            raise ValueError(\'Invalid optimization source record\')\n        raw = base64.b64decode(item[\'data\'], validate=True)\n        if sha256(raw).hexdigest() != item[\'sha256\']:\n            raise ValueError(\'Optimization source hash differs: \' + name)\n        compile(raw, name, \'exec\')\n        result[name] = raw\n    return result\n\ndef stage_overlay(work, payload):\n    work = Path(work)\n    validated(payload)\n    target = work / \'optimized-worker.json\'\n    with target.open(\'x\', encoding=\'utf-8\') as file:\n        json.dump(payload, file)\n    return target\n\ndef install_overlay(root, payload_file, output):\n    root, output = Path(root), Path(output)\n    for port in (8765, 8800):\n        with socket.socket() as connection:\n            connection.settimeout(1)\n            if connection.connect_ex((\'127.0.0.1\', port)) == 0:\n                raise RuntimeError(\'Do not replace source while an AI service is running\')\n    record = output / \'optimized-worker-release.json\'\n    if record.exists():\n        raise RuntimeError(\'Optimization already installed; use a fresh session\')\n    payload = json.loads(Path(payload_file).read_text(encoding=\'utf-8\'))\n    sources = validated(payload)\n    manifest_file = root / \'gate3_bundle.json\'\n    manifest = json.loads(manifest_file.read_text(encoding=\'utf-8\'))\n    # The original bootstrap verifies the entire bundle first. Check every replaced\n    # file again here; never rewrite that pinned manifest or private archive.\n    for name in FILES:\n        target = root / name\n        if name in manifest[\'files\']:\n            if sha256(target.read_bytes()).hexdigest() != manifest[\'files\'][name]:\n                raise ValueError(\'Original worker source differs before overlay\')\n        elif target.exists():\n            raise ValueError(\'Unexpected preexisting optimization source\')\n    for name, raw in sources.items():\n        target = root / name\n        target.parent.mkdir(parents=True, exist_ok=True)\n        target.write_bytes(raw)\n    evidence = {\'policy\': \'nails-only\', \'original_manifest_sha256\': sha256(manifest_file.read_bytes()).hexdigest(),\n                \'source_sha256\': {name: sha256(raw).hexdigest() for name, raw in sources.items()},\n                \'model_parameters_changed\': False}\n    record.write_text(json.dumps(evidence, indent=2), encoding=\'utf-8\')\n    print(\'OPTIMIZATION: Nails-only public-prompt reuse; Hair/Makeup use baseline inference.\', flush=True)\n    return evidence\n', encoding='utf-8')
print('Optimized worker staged. Run Cell 2 once.')


In [ ]:
import os
BEAUTYCORE_PUBLIC_URL = 'https://beautycore-demo.onrender.com'
os.environ['BEAUTYCORE_PUBLIC_URL'] = BEAUTYCORE_PUBLIC_URL
(CLIENT_WORK/'runtime_registration.py').write_text('"""HMAC metadata registration/heartbeat. No credentials or generation requests in logs."""\nfrom hashlib import sha256\nimport hmac\nimport json\nimport os\nfrom pathlib import Path\nimport socket\nimport sys\nimport time\nfrom urllib.error import HTTPError, URLError\nfrom urllib.parse import urlsplit\nfrom urllib.request import Request, HTTPRedirectHandler, build_opener\n\nSCHEMA = "beautycore-runtime-v1"\nHEARTBEAT_SECONDS = 60\n\nclass RegistrationError(RuntimeError):\n    def __init__(self, status=None):\n        self.status = status\n        super().__init__("Website connection not acknowledged" + (f" (HTTP {status})" if status else ""))\n\nclass NoRedirect(HTTPRedirectHandler):\n    def redirect_request(self, req, fp, code, msg, headers, newurl):\n        return None\n\ndef canonical(message):\n    return json.dumps(message, sort_keys=True, separators=(",", ":"), ensure_ascii=True).encode()\n\ndef sign(message, key):\n    derived = hmac.new(key.encode(), b"beautycore-runtime-key-v1", sha256).digest()\n    return hmac.new(derived, canonical(message), sha256).hexdigest()\n\ndef website_root(website):\n    p = urlsplit(website)\n    if (p.scheme != "https" or not p.hostname or p.username or p.password or p.port\n            or p.query or p.fragment or p.path.rstrip("/")):\n        raise RegistrationError()\n    return website.rstrip("/")\n\ndef send(website, record, key, operation, *, opener=None, now=time.time):\n    if len(key) < 32:\n        raise RegistrationError()\n    message = {"schema": SCHEMA, "operation": operation, "session_id": record["session_id"],\n               "endpoint": record["url"], "started_at": record["started_at"], "issued_at": int(now())}\n    request = Request(website_root(website) + "/api/internal/ai-runtime/register", data=canonical(message),\n                      headers={"Content-Type": "application/json", "x-ai-runtime-signature": sign(message, key)})\n    try:\n        with (opener or build_opener(NoRedirect())).open(request, timeout=75) as response:\n            if response.status != 200:\n                raise RegistrationError(response.status)\n            raw = response.read(8193)\n            if len(raw) > 8192:\n                raise RegistrationError()\n            ack = json.loads(raw)\n            if (ack.get("connected") is not True or ack.get("session_id") != record["session_id"]\n                    or not isinstance(ack.get("expires_at"), (int, float)) or ack["expires_at"] <= now() * 1000):\n                raise RegistrationError()\n            return ack\n    except HTTPError as error:\n        raise RegistrationError(error.code) from None\n    except (URLError, OSError, TimeoutError, ValueError, TypeError, AttributeError):\n        raise RegistrationError() from None\n\ndef register(website, record, key, *, send_fn=send, sleep=time.sleep, attempts=4):\n    # Metadata only. The GPU request is never repeated here. Accommodates Render cold start.\n    for attempt in range(attempts):\n        try:\n            return send_fn(website, record, key, "register")\n        except RegistrationError as error:\n            if error.status in (400, 401, 403, 409, 404) or attempt == attempts - 1:\n                raise\n            print("Website registration waiting; AI processes remain running.", flush=True)\n            sleep(10)\n\ndef services_alive(record):\n    try:\n        for field in ("api_pid", "tunnel_pid"):\n            os.kill(record[field], 0)\n        for port in (8765, 8800):\n            with socket.socket() as connection:\n                connection.settimeout(2)\n                if connection.connect_ex(("127.0.0.1", port)) != 0:\n                    return False\n        return True\n    except (OSError, KeyError, TypeError):\n        return False\n\ndef heartbeat(website, record, key, *, send_fn=send, alive=services_alive, sleep=time.sleep, now=time.time):\n    acknowledged_until = record["registration_expires_at"]\n    while True:\n        sleep(HEARTBEAT_SECONDS)\n        if not alive(record):\n            print("AI process stopped. Website lease will expire; start a fresh notebook session.", flush=True)\n            return\n        try:\n            ack = send_fn(website, record, key, "renew")\n            acknowledged_until = ack["expires_at"]\n        except RegistrationError as error:\n            if error.status in (400, 401, 403, 409, 404) or now() * 1000 >= acknowledged_until:\n                print("Website lease expired or refused. Start a fresh session; do not replay generation.", flush=True)\n                return\n            print("Website heartbeat not acknowledged; existing lease has not been extended.", flush=True)\n\nif __name__ == "__main__":\n    record = json.loads(Path(sys.argv[2]).read_text())\n    heartbeat(sys.argv[1], record, os.environ.get("AI_BACKEND_API_KEY", ""))\n', encoding='utf-8')
"""New scheduled-demo runner. Existing notebooks and GPU bundle are not modified."""
import argparse
import json
import os
from pathlib import Path
import re
import socket
import subprocess
import sys
import time
from uuid import uuid4
from urllib.request import Request, urlopen

WORK = CLIENT_WORK
GPU_ROOT = WORK / "gpu-src"
CPU_ROOT = WORK / "cpu-src"
PORT = 8800


def command(args, cwd, env, stage):
    print("DEMO STAGE:", stage, flush=True)
    log = WORK / (stage + ".log")
    with log.open("w") as output:
        process = subprocess.Popen(args, cwd=cwd, env=env, stdout=output, stderr=subprocess.STDOUT)
        while process.poll() is None:
            try:
                process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                print(stage, "still running; log bytes:", log.stat().st_size, flush=True)
    if process.returncode:
        raise RuntimeError(stage + " failed; inspect its log")


def read_json(url, key=None):
    headers = {"x-ai-backend-key": key, "x-ai-user-id": "demo-operator"} if key else {}
    with urlopen(Request(url, headers=headers), timeout=40) as response:
        return json.load(response)


def main():
    if sys.platform != "linux":
        raise RuntimeError("Run in the new Kaggle Linux GPU notebook")
    if (WORK / "demo-session.json").exists():
        raise RuntimeError("Refuse duplicate startup; use a fresh notebook session")
    for port in (8765, PORT):
        with socket.socket() as connection:
            connection.settimeout(1)
            if connection.connect_ex(("127.0.0.1", port)) == 0:
                raise RuntimeError("An AI service is already running; do not start another worker")
    # Total RAM, not VRAM. Old GPU evidence peaked near 23 GiB before CPU vision.
    memory = int(re.search(r"MemTotal:\s+(\d+)", Path("/proc/meminfo").read_text())[1]) * 1024
    if memory < 30 * 1024**3:
        raise RuntimeError("At least 30 GiB host RAM required for this unverified combined workload")
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    backend_key = secrets.get_secret("AI_BACKEND_API_KEY") or ""
    gpu_key = secrets.get_secret("AI_REMOTE_API_KEY") or ""
    gemini_key = secrets.get_secret("GEMINI_API_KEY") or ""
    if len(backend_key) < 32 or len(gpu_key) < 24 or not gemini_key:
        raise RuntimeError("Enable the three required Kaggle Secrets; never put keys into cells")
    session_id = str(uuid4())
    started_at = int(time.time())
    # Reserve the session before starting any subprocess; failed sessions need a fresh restart.
    (WORK / "demo-session.json").write_text(json.dumps({"status": "starting", "host_ram_bytes": memory}))
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    # CPU services do not use Kaggle's inline-notebook plotting backend.
    cpu_env = {**env, "CUDA_VISIBLE_DEVICES": "", "OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1", "MPLBACKEND": "Agg"}
    import ctypes.util
    if not ctypes.util.find_library("portaudio"):
        command(["apt-get", "update"], CPU_ROOT, env, "portaudio-index")
        command(["apt-get", "install", "-y", "libportaudio2"], CPU_ROOT, env, "portaudio-install")
    for name in ("backend-env", "segment-env"):
        # Kaggle ensurepip can fail even on the reviewed Python image.
        # Use host pip only as an installer targeting the isolated CPU environment.
        command([sys.executable, "-m", "venv", "--without-pip", str(WORK / name)], CPU_ROOT, cpu_env, name + "-create")
        command([sys.executable, "-m", "pip", "--python", str(WORK / name), "install", "pip==25.3"],
                CPU_ROOT, cpu_env, name + "-pip-bootstrap")
    api_python = WORK / "backend-env/bin/python"
    seg_python = WORK / "segment-env/bin/python"
    command([str(api_python), "-m", "pip", "install", "-r", "backend/requirements-nails.txt", "--constraint", "backend/requirements-demo-constraints.txt"],
            CPU_ROOT, cpu_env, "backend-install")
    command([str(seg_python), "-m", "pip", "install", "torch==2.10.0", "torchvision==0.25.0", "--index-url", "https://download.pytorch.org/whl/cpu"],
            CPU_ROOT, cpu_env, "segment-torch-install")
    command([str(seg_python), "-m", "pip", "install", "-r", "backend/requirements-nails-segmenter.txt"],
            CPU_ROOT, cpu_env, "segment-install")
    for name, interpreter in (("backend", api_python), ("segment", seg_python)):
        command([str(interpreter), "-m", "pip", "check"], CPU_ROOT, cpu_env, name + "-pip-check")
    checkpoint = CPU_ROOT / "data/nails/checkpoints/mnemic/nails_seg_s_yolov8_v1.pt"
    command([str(seg_python), "-c", "import torch, torchvision; from ultralytics import YOLO; assert torchvision.extension._has_ops(); assert '+cpu' in torch.__version__; YOLO(" + repr(str(checkpoint)) + ")"],
            CPU_ROOT, cpu_env, "segment-load")
    landmark = CPU_ROOT / "data/nails/checkpoints/mediapipe/hand_landmarker.task"
    cpu_env["PYTHONPATH"] = str(CPU_ROOT / "backend")
    command([str(api_python), "-c", "from pathlib import Path; from app.nails.geometry import MediaPipeHandLocalizer; MediaPipeHandLocalizer(Path(" + repr(str(landmark)) + "))"],
            CPU_ROOT, cpu_env, "landmark-load")
    gpu_code = f"""from pathlib import Path
from scripts import unified_gate3_bootstrap as original
from scripts.capstone_train002_bootstrap import worker_environment
start = original.start_server
import importlib.util
spec = importlib.util.spec_from_file_location('optimized_overlay', Path({str(WORK / 'worker_overlay.py')!r}))
release_overlay = importlib.util.module_from_spec(spec)
spec.loader.exec_module(release_overlay)
def start_optimized(output, env):
    original.verify_bundle()
    release_overlay.install_overlay(original.ROOT, Path({str(WORK / 'optimized-worker.json')!r}), output)
    return start(output, worker_environment(dict(env, AI_PROMPT_CACHE='1', AI_PROMPT_CACHE_SCOPE='nails', AI_INFERENCE_PROFILE='0')))
original.start_server = start_optimized
report = original.bootstrap(Path({str(WORK / 'gpu-session')!r}), local_only=True)
assert report['status'] == 'READY_FOR_LIVE_ACCEPTANCE', 'GPU bootstrap failed'
"""
    # Preserves exact host/package/model/adapter checks; one GPU load, no public GPU tunnel.
    command([sys.executable, "-c", gpu_code], GPU_ROOT,
            {**env, "AI_REMOTE_API_KEY": gpu_key}, "gpu-bootstrap")
    cpu_env.update(AI_BACKEND_API_KEY=backend_key, AI_GPU_LOOPBACK="1",
                   AI_REMOTE_URL="http://127.0.0.1:8765", AI_REMOTE_API_KEY=gpu_key,
                   GENERATION_ENGINE="remote_flux", MAKEUP_GENERATION_ENGINE="remote_makeup",
                   NAILS_PREVIEW_MODE="hybrid", NAILS_INFERENCE_STEPS="8",
                   NAILS_HAND_LANDMARKER_PATH=str(landmark), NAILS_SEGMENT_CHECKPOINT=str(checkpoint),
                   NAILS_SEGMENT_PYTHON=str(seg_python), CONSULTATION_PROVIDER="gemini",
                   CONSULTATION_GEMINI_MODEL="gemini-3.5-flash-lite", GEMINI_API_KEY=gemini_key,
                   HAIRCAPSTONE_STYLE_REGISTRY_PATH=str(CPU_ROOT / "backend/app/style_registry_train002_smoke.json"),
                   FRONTEND_ORIGINS="")
    with (WORK / "api.log").open("w") as log:
        api = subprocess.Popen([str(api_python), "-m", "uvicorn", "app.demo_server:app",
                                "--host", "127.0.0.1", "--port", str(PORT), "--workers", "1"],
                               cwd=CPU_ROOT, env=cpu_env, stdout=log, stderr=subprocess.STDOUT,
                               start_new_session=True)
    deadline = time.monotonic() + 120
    while True:
        if api.poll() is not None:
            raise RuntimeError("API exited; inspect api.log")
        try:
            if read_json(f"http://127.0.0.1:{PORT}/health")["status"] == "ok":
                break
        except Exception:
            pass
        if time.monotonic() >= deadline:
            raise RuntimeError("API did not start; inspect api.log")
        time.sleep(2)
    readiness = read_json(f"http://127.0.0.1:{PORT}/deployment/readiness", backend_key)
    if not readiness.get("ready"):
        raise RuntimeError("AI readiness failed; inspect API and GPU logs")
    # Bootstrap already downloaded cloudflared; point only at the authenticated facade.
    binary = WORK / "gpu-session/cloudflared"
    if not binary.exists():
        import shutil
        with urlopen("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", timeout=90) as source, binary.open("wb") as target:
            shutil.copyfileobj(source, target)
        binary.chmod(0o700)
    with (WORK / "tunnel.log").open("w") as log:
        tunnel = subprocess.Popen([str(binary), "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
                                  cwd=WORK, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    deadline = time.monotonic() + 240
    while True:
        if tunnel.poll() is not None:
            raise RuntimeError("API tunnel exited; inspect tunnel.log")
        match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", (WORK / "tunnel.log").read_text())
        if match:
            try:
                if read_json(match[0] + "/deployment/readiness", backend_key).get("ready"):
                    break
            except Exception:
                pass
        if time.monotonic() >= deadline:
            raise RuntimeError("Public API readiness failed; inspect tunnel.log")
        time.sleep(3)
    record = {"status": "READY_FOR_REHEARSAL", "url": match[0], "api_pid": api.pid,
              "tunnel_pid": tunnel.pid, "readiness": readiness, "host_ram_bytes": memory,
              "full_generation_verified": False}
    (WORK / "demo-session.json").write_text(json.dumps(record, indent=2))
    website = os.environ.get("BEAUTYCORE_PUBLIC_URL", "").strip()
    if website:
        import importlib.util
        registrar_path = WORK / "runtime_registration.py"
        if not registrar_path.is_file():
            raise RuntimeError("Automatic registration helper is missing; use the corrected client notebook")
        spec = importlib.util.spec_from_file_location("demo_registrar", registrar_path)
        registrar = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(registrar)
        record.update(session_id=session_id, started_at=started_at, status="AI_READY_CONNECTION_PENDING")
        (WORK / "demo-session.json").write_text(json.dumps(record, indent=2))
        try:
            ack = registrar.register(website, record, backend_key)
        except registrar.RegistrationError:
            print("AI is running, but website registration was not acknowledged. Inspect registration status; do not rerun GPU startup.", flush=True)
            raise
        record.update(status="CONNECTED_FOR_REHEARSAL", registration_expires_at=ack["expires_at"])
        (WORK / "demo-session.json").write_text(json.dumps(record, indent=2))
        with (WORK / "registration.log").open("w") as log:
            heartbeat = subprocess.Popen([sys.executable, str(registrar_path), website, str(WORK / "demo-session.json")],
                env={**env, "AI_BACKEND_API_KEY": backend_key}, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        record["heartbeat_pid"] = heartbeat.pid
        (WORK / "demo-session.json").write_text(json.dumps(record, indent=2))
        print("CONNECTED TO WEBSITE:", website, flush=True)
        print("No Render URL edit is needed. Keep the Kaggle session running.", flush=True)
    else:
        print("Render AI_FASTAPI_URL:", match[0], flush=True)
    print("READY_FOR_REHEARSAL: run the full application checklist before the defense.", flush=True)



main()


In [ ]:
record=json.loads((CLIENT_WORK/'demo-session.json').read_text())
print('Status:',record['status'])
print('Website:',BEAUTYCORE_PUBLIC_URL)
print('Optimization: Nails-only; Hair/Makeup baseline')
print('Rehearse Hair, Makeup, model Nails, renderer Nails and Consultation.')
